# Chronos only — zero-shot foundation-model forecasts for the paper

This notebook runs **only** the Chronos models (the part that cannot run on Claude's side because the models download from Hugging Face). It uses exactly the same data loading, split and forecast origins as the full analysis, saves every Chronos forecast to `Downloads/chronos_results`, and shows a quick comparison with persistence after each run.

**Before you start**, run this once in a cell or in Terminal: `pip install chronos-forecasting torch`

Run top to bottom. Finished runs are cached: if you stop, rerunning skips them. At the end, zip `Downloads/chronos_results` (without the `cache` folder) and send it to Claude.

In [ ]:
# ===================================================================
# STEP 1 — SETTINGS (edit the paths if your files are elsewhere)
# ===================================================================
import os, time, json, copy, random, warnings, pickle, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import stats
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

HOME = Path.home()
DATA = HOME / "Downloads" / "open-forecasting-framework-v2" / "data" / "raw"
ISONE_FILES = [DATA / "isone" / f"{y}_smd_hourly.xlsx" for y in (2023, 2024, 2025)]
TETOUAN_CSV = DATA / "power_consumption.csv"
EPF_DIR = HOME / "Downloads" / "epf_data"               # official EPF files are downloaded here (STEP 23)
OUT = HOME / "Downloads" / "chronos_results"            # Chronos forecasts are saved here

DEVICE = "mps"           # "mps" on Apple silicon, "cuda" on NVIDIA, "cpu" otherwise
SEED = 42
H = 24                   # forecast horizon (hours)

(OUT / "cache").mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(parents=True, exist_ok=True)
for f in ISONE_FILES + [TETOUAN_CSV]:
    print("OK     " if f.exists() else "MISSING", f)

def cached(name, compute):
    """Run compute() once and save the result; later runs load it from disk."""
    path = OUT / "cache" / f"{name}.pkl"
    if path.exists():
        with open(path, "rb") as fh:
            print(f"[loaded from cache] {name}")
            return pickle.load(fh)
    start = time.time()
    result = compute()
    with open(path, "wb") as fh:
        pickle.dump(result, fh)
    print(f"[computed in {(time.time() - start) / 60:.1f} min] {name}")
    return result

In [ ]:
# ===================================================================
# STEP 2 — LOAD ISO NEW ENGLAND ZONAL LOAD AND WEATHER (main dataset)
# Declared daylight-saving policy:
#  * Hr_End is hour-ending local time. 2024-2025 files follow the true clock
#    (23-h day in March, 25-h day in November with "02X").
#  * 2023 uses a legacy 24-rows-per-day convention: the synthetic spring HE02 is
#    dropped; the averaged autumn HE02 is kept and the second hour interpolated.
#  * Timestamps: hour-beginning on a fixed UTC-05:00 clock (continuous index).
# ===================================================================
ZONES = ["ME", "NH", "VT", "CT", "RI", "SEMA", "WCMA", "NEMA"]
LOCAL_TZ, FIXED_OFFSET = "America/New_York", pd.Timedelta(hours=-5)

def load_isone_zone(path, zone):
    frame = pd.read_excel(path, sheet_name=zone, usecols=["Date", "Hr_End", "RT_Demand", "Dry_Bulb", "Dew_Point"])
    frame["h"] = frame["Hr_End"].astype(str).str.strip()
    frame["Date"] = pd.to_datetime(frame["Date"])
    counts = {"spring_dropped": 0, "autumn_interpolated": 0}
    days = []
    for day, g in frame.groupby("Date", sort=True):
        g = g.copy()
        start = pd.Timestamp(day).tz_localize(LOCAL_TZ)
        end = pd.Timestamp(day + pd.Timedelta(days=1)).tz_localize(LOCAL_TZ)
        real = int((end - start) / pd.Timedelta(hours=1))
        if len(g) == 24 and real == 23:
            g = g[g["h"].str.lstrip("0") != "2"]; counts["spring_dropped"] += 1
        if len(g) == 24 and real == 25:
            pos = g.index[g["h"].str.lstrip("0") == "2"][0]
            gap = g.loc[[pos]].copy(); gap[["RT_Demand", "Dry_Bulb", "Dew_Point"]] = np.nan
            g = pd.concat([g.loc[:pos], gap, g.loc[pos + 1:]]); counts["autumn_interpolated"] += 1
        assert len(g) == real, f"{path.name} {zone} {day.date()}: {len(g)} rows for {real} hours"
        utc = start.tz_convert("UTC") + pd.to_timedelta(np.arange(real), unit="h")
        g.index = (utc + FIXED_OFFSET).tz_localize(None)
        days.append(g[["RT_Demand", "Dry_Bulb", "Dew_Point"]])
    return pd.concat(days), counts

def load_isone_load():
    columns, audit = {}, {}
    for zone in ZONES:
        parts = []
        for path in ISONE_FILES:
            data, c = load_isone_zone(path, zone); parts.append(data)
            for k, v in c.items(): audit[k] = audit.get(k, 0) + v
        z = pd.concat(parts)
        columns[zone] = z["RT_Demand"]
        columns[f"{zone}_Dry_Bulb"] = z["Dry_Bulb"]; columns[f"{zone}_Dew_Point"] = z["Dew_Point"]
    hourly = pd.DataFrame(columns)
    hourly = hourly.reindex(pd.date_range(hourly.index.min(), hourly.index.max(), freq="h"))
    audit["cells_interpolated"] = int(hourly.isna().sum().sum())
    return hourly.interpolate(limit=2), audit

isone, isone_audit = cached("data_isone_load", load_isone_load)
ISONE_WEATHER = [f"{z}_{v}" for z in ZONES for v in ("Dry_Bulb", "Dew_Point")]
print("Shape:", isone.shape, "| from", isone.index.min(), "to", isone.index.max())
print("Daylight-saving audit (all zones):", isone_audit)
display(isone[ZONES].describe().round(1))

In [ ]:
# ===================================================================
# STEP 3 — LOAD ISO-NE SYSTEM DAY-AHEAD PRICE
# Price days use 24 local wall-clock hours (EPF convention): the missing spring
# hour is interpolated, the repeated autumn hour (02 and 02X) is averaged.
# DA_Demand is NOT loaded: it is cleared by the same auction as the price.
# ===================================================================
def load_isone_price():
    cols = ["DA_LMP", "RT_Demand", "Dry_Bulb", "Dew_Point"]
    f = pd.concat([pd.read_excel(p, sheet_name="ISO NE CA", usecols=["Date", "Hr_End", *cols]) for p in ISONE_FILES], ignore_index=True)
    he = pd.to_numeric(f["Hr_End"].astype(str).str.strip().str.replace("X", ""))
    f["ts"] = pd.to_datetime(f["Date"]) + pd.to_timedelta(he - 1, unit="h")
    hourly = f.groupby("ts")[cols].mean()
    hourly = hourly.reindex(pd.date_range(hourly.index.min(), hourly.index.max(), freq="h")).interpolate(limit=2)
    return hourly.rename(columns={"DA_LMP": "Price"})

isone_price = cached("data_isone_price", load_isone_price)
p = isone_price["Price"]
print("Hourly DA LMP ($/MWh): mean %.2f | median %.2f | 99th pct %.2f | max %.2f | hours > $100: %.1f%%"
      % (p.mean(), p.median(), p.quantile(0.99), p.max(), 100 * (p > 100).mean()))
display(isone_price.head())

In [ ]:
# ===================================================================
# STEP 4 — LOAD TETOUAN (secondary dataset)
# 10-minute data -> hourly means; only complete hours are allowed.
# The source does not state a unit: values are kept "as published".
# ===================================================================
TET_ZONES = ["Zone 1", "Zone 2"]
TET_WEATHER = ["Temperature", "Humidity", "Wind Speed", "general diffuse flows", "diffuse flows"]

def load_tetouan():
    raw = pd.read_csv(TETOUAN_CSV)
    raw.columns = [" ".join(str(c).split()).replace(" Power Consumption", "") for c in raw.columns]
    text = raw["DateTime"].astype(str).str.strip()
    idx = pd.to_datetime(text, format="%m/%d/%Y %H:%M", errors="coerce")
    idx = idx.fillna(pd.to_datetime(text, format="%m-%d-%Y %H:%M", errors="coerce"))
    assert idx.notna().all() and idx.is_monotonic_increasing and not idx.duplicated().any()
    data = raw[TET_ZONES + TET_WEATHER].set_index(pd.DatetimeIndex(idx))
    counts = data.resample("h").count()
    assert (counts == 6).all().all(), "incomplete hours found"
    return data.resample("h").mean()

tetouan = cached("data_tetouan", load_tetouan)
print("Shape:", tetouan.shape, "| from", tetouan.index.min(), "to", tetouan.index.max())
display(tetouan.describe().round(2))

In [ ]:
# ===================================================================
# STEP 6 — INFORMATION SETS AND HORIZON-PURGED SPLIT
#  A  weather_time                : weather at t+H (declared proxy) + calendar of t+H
#  B  weather_time_history        : A + load lags at t-1,2,3,6,12,24 h + 3 h / 24 h rolling means
#  B+ weather_time_history_weekly : B + load at t-144 h and t-168 h
# ===================================================================
MODES = ["weather_time", "weather_time_history", "weather_time_history_weekly"]
MODE_NAME = {"weather_time": "A", "weather_time_history": "B", "weather_time_history_weekly": "B+"}
LAGS, ROLLS, WEEKLY = [1, 2, 3, 6, 12, 24], [3, 24], [144, 168]

def calendar(index):
    return pd.DataFrame({"hour_sin": np.sin(2 * np.pi * index.hour / 24), "hour_cos": np.cos(2 * np.pi * index.hour / 24),
                         "dow_sin": np.sin(2 * np.pi * index.dayofweek / 7), "dow_cos": np.cos(2 * np.pi * index.dayofweek / 7),
                         "is_weekend": (index.dayofweek >= 5).astype(float)}, index=index)

def tabular(hourly, targets, weather, mode):
    x = hourly[weather].shift(-H).add_prefix("target_weather_")
    cal = calendar(hourly.index + pd.Timedelta(hours=H)); cal.index = hourly.index
    x = pd.concat([x, cal], axis=1)
    if mode != "weather_time":
        for z in targets:
            for lag in LAGS: x[f"{z}_lag_{lag}h"] = hourly[z].shift(lag)
            for w in ROLLS: x[f"{z}_roll_{w}h"] = hourly[z].rolling(w).mean()
            if mode == "weather_time_history_weekly":
                for lag in WEEKLY: x[f"{z}_weekly_{lag}h"] = hourly[z].shift(lag)
    y = hourly[targets].shift(-H)
    ok = x.notna().all(axis=1) & y.notna().all(axis=1)
    return x.loc[ok], y.loc[ok]

def purged_split(index, train_frac, val_frac):
    n = len(index); tr = int(n * train_frac); ve = tr + int(n * val_frac)
    return {"train": index[:tr], "validation": index[tr + H:ve], "test": index[ve + H:]}

def prepare(hourly, targets, weather, train_frac, val_frac):
    frames = {m: tabular(hourly, targets, weather, m) for m in MODES}
    common = frames[MODES[0]][0].index
    for m in MODES[1:]: common = common.intersection(frames[m][0].index)
    frames = {m: (x.loc[common], y.loc[common]) for m, (x, y) in frames.items()}
    return frames, purged_split(common, train_frac, val_frac)

iso_frames, iso_split = prepare(isone, ZONES, ISONE_WEATHER, 0.5, 0.1667)
tet_frames, tet_split = prepare(tetouan, TET_ZONES, TET_WEATHER, 0.7, 0.15)
for name, sp in [("ISO-NE", iso_split), ("Tetouan", tet_split)]:
    print(f"\n{name} split (forecast origins, {H}-h purge between partitions):")
    display(pd.DataFrame({k: [len(v), v.min(), v.max()] for k, v in sp.items()}, index=["origins", "first", "last"]).T)
    assert sp["train"].max() + pd.Timedelta(hours=H) < sp["validation"].min()
    assert sp["validation"].max() + pd.Timedelta(hours=H) < sp["test"].min()
print("Purge check passed. Features per mode (ISO-NE):", {MODE_NAME[m]: f.shape[1] for m, (f, _) in iso_frames.items()})

In [ ]:
# ===================================================================
# STEP 7 — CHRONOS HELPERS (identical to the full analysis notebook)
# ===================================================================
FOUNDATION = {"chronos_bolt_small": "amazon/chronos-bolt-small", "chronos_bolt_base": "amazon/chronos-bolt-base", "chronos2": "amazon/chronos-2"}
_pipes = {}
def chronos(name):
    if name not in _pipes:
        import torch
        from chronos import BaseChronosPipeline
        _pipes[name] = BaseChronosPipeline.from_pretrained(FOUNDATION[name], device_map=DEVICE, torch_dtype=torch.float32)
    return _pipes[name]

def chronos_median(name, contexts, horizon, past=None, future=None, batch=256):
    import torch
    pipe, out = chronos(name), []
    for s in range(0, len(contexts), batch):
        if name == "chronos2":
            items = [{"target": contexts[i].astype(np.float32),
                      **({"past_covariates": {k: v.astype(np.float32) for k, v in past[i].items()},
                          "future_covariates": {k: v.astype(np.float32) for k, v in future[i].items()}} if past else {})}
                     for i in range(s, min(s + batch, len(contexts)))]
            q, _ = pipe.predict_quantiles(items, prediction_length=horizon, quantile_levels=[0.5])
            out += [np.asarray(t.float().cpu())[0, :, 0] for t in q]
        else:
            q, _ = pipe.predict_quantiles([torch.tensor(c, dtype=torch.float32) for c in contexts[s:s + batch]], prediction_length=horizon, quantile_levels=[0.5])
            out += list(np.asarray(q.float().cpu())[:, :, 0])
    return np.vstack(out)

def fm_load(hourly, targets, weather, split, name, context=512):
    origins = split["test"]; pos = hourly.index.get_indexer(origins)
    pred = np.zeros((len(origins), len(targets)))
    for j, z in enumerate(targets):
        series = hourly[z].to_numpy(float)
        zw = [c for c in weather if c.startswith(f"{z}_")] or list(weather)
        ctx = [series[max(0, p - context + 1):p + 1] for p in pos]
        past = fut = None
        if name == "chronos2":
            past = [{c: hourly[c].to_numpy()[max(0, p - context + 1):p + 1] for c in zw} for p in pos]
            fut = [{c: hourly[c].to_numpy()[p + 1:p + 1 + H] for c in zw} for p in pos]
        pred[:, j] = chronos_median(name, ctx, H, past, fut)[:, H - 1]
        print(f"   {name}: zone {z} done")
    return hourly[targets].shift(-H).loc[origins].to_numpy(), pred


def daily_matrix(series):
    f = series.to_frame("v"); f["day"] = f.index.normalize(); f["hour"] = f.index.hour
    m = f.pivot_table(index="day", columns="hour", values="v", aggfunc="mean")
    return m[m.notna().sum(axis=1) == 24]

def price_design(hourly, exogenous_lags, info):
    price = daily_matrix(hourly["Price"]); days = pd.date_range(price.index.min(), price.index.max(), freq="D")
    price = price.reindex(days); blocks = []
    for lag in (1, 2, 3, 7):
        b = price.shift(lag); b.columns = [f"price_d{lag}_h{h}" for h in range(24)]; blocks.append(b)
    if info == "price_exogenous":
        for col, lags in exogenous_lags.items():
            m = daily_matrix(hourly[col]).reindex(days)
            for lag in lags:
                b = m.shift(lag); b.columns = [f"{col}_d{lag}_h{h}" for h in range(24)]; blocks.append(b)
    dow = pd.get_dummies(pd.Series(days.dayofweek, index=days), prefix="dow").astype(float)
    x = pd.concat([*blocks, dow], axis=1); y = price.copy(); y.columns = [f"h{h}" for h in range(24)]
    ok = x.notna().all(axis=1) & y.notna().all(axis=1)
    return x.loc[ok], y.loc[ok]


def fm_price(hourly, days, name, exogenous_lags, use_exog, context=2048):
    price = hourly["Price"].to_numpy(float); ctx, past, fut = [], [], []
    for d in days:
        e = hourly.index.get_indexer([d])[0]; lo = max(0, e - context)
        ctx.append(price[lo:e])
        if use_exog:
            past.append({c: hourly[c].to_numpy()[lo:e] for c in exogenous_lags})
            fut.append({c: hourly[c].to_numpy()[e:e + 24] for c, lags in exogenous_lags.items() if 0 in lags})
    return chronos_median(name, ctx, 24, past or None, fut or None, batch=64)


(OUT / "forecasts").mkdir(exist_ok=True)
def save(name, index, values, columns):
    pd.DataFrame(values, index=index, columns=columns).to_csv(OUT / "forecasts" / f"{name}.csv")

def quick_check(actual, pred, bench, label):
    rmse = lambda p: np.sqrt(((actual - p) ** 2).mean())
    print(f"   {label}: RMSE {rmse(pred):.1f} vs persistence {rmse(bench):.1f} -> ratio {rmse(pred) / rmse(bench):.3f}")

In [ ]:
# ===================================================================
# STEP 8 — CHRONOS ON ISO-NE LOAD (8 zones, 8,680 test origins; about 1-2 h)
# ===================================================================
iso_test = iso_split["test"]
iso_actual = isone[ZONES].shift(-H).loc[iso_test].to_numpy()
iso_persist = isone.loc[iso_test, ZONES].to_numpy()
for name in FOUNDATION:
    actual, pred = cached(f"isone_{name}", lambda: fm_load(isone, ZONES, ISONE_WEATHER, iso_split, name))
    save(f"isone_load_{name}", iso_test, pred, ZONES)
    quick_check(iso_actual, pred, iso_persist, f"ISO-NE {name}")

In [ ]:
# ===================================================================
# STEP 9 — CHRONOS ON TETOUAN LOAD (2 zones, 1,259 test origins; minutes)
# ===================================================================
tet_test = tet_split["test"]
tet_actual = tetouan[TET_ZONES].shift(-H).loc[tet_test].to_numpy()
tet_persist = tetouan.loc[tet_test, TET_ZONES].to_numpy()
for name in FOUNDATION:
    actual, pred = cached(f"tetouan_{name}", lambda: fm_load(tetouan, TET_ZONES, TET_WEATHER, tet_split, name))
    save(f"tetouan_load_{name}", tet_test, pred, TET_ZONES)
    quick_check(tet_actual, pred, tet_persist, f"Tetouan {name}")

In [ ]:
# ===================================================================
# STEP 10 — CHRONOS ON ISO-NE DAY-AHEAD PRICE (365 days of 2025; minutes)
# Chronos-2 is also run with the exogenous inputs (weather at D as future
# covariate, load only as a past covariate).
# ===================================================================
ISONE_EXOG = {"Dry_Bulb": (0,), "Dew_Point": (0,), "RT_Demand": (1, 7)}
_, y_price = price_design(isone_price, ISONE_EXOG, "price_only")
test_days = y_price.index[(y_price.index >= "2025-01-01") & (y_price.index <= "2025-12-31")]
truth = y_price.loc[test_days].to_numpy()
for name in FOUNDATION:
    for info in (["price_only", "price_exogenous"] if name == "chronos2" else ["price_only"]):
        pred = cached(f"price_isone_{name}_{info}", lambda: fm_price(isone_price, test_days, name, ISONE_EXOG, info == "price_exogenous"))
        save(f"isone_price_{name}_{info}", test_days, pred, [f"h{h}" for h in range(24)])
        print(f"   ISO-NE price {name} | {info}: MAE {np.abs(truth - pred).mean():.2f} $/MWh (LEAR with exogenous inputs: 12.21)")

In [ ]:
# ===================================================================
# STEP 11 — ZIP FOR CLAUDE
# ===================================================================
import zipfile
out = HOME / "Downloads" / "chronos_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in (OUT / "forecasts").glob("*.csv"):
        z.write(f, f.name)
print("Send this file to Claude:", out, f"({out.stat().st_size / 1e6:.1f} MB)")